# Alleneremo un LLM per generare musica Jazz like

Definiamo un helper per processare l'audio

In [121]:
from music21 import converter, instrument, note, chord
import glob

def get_notes_from_midi(folder_path):
    notes = []

    # Cerchiamo tutti i file .mid nella cartella
    for file in glob.glob(f"{folder_path}/*.mid"):
        midi = converter.parse(file)
        print(f"Parsing: {file}")

        notes_to_parse = None

        # Proviamo a dividere per strumenti
        try:
            s2 = instrument.partitionByInstrument(midi)
            # Spesso nel jazz vogliamo solo il piano o il solista
            notes_to_parse = s2.parts[0].recurse()
        except:
            # Se non ci sono parti definite, prendiamo tutto
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            # Se l'elemento è una nota singola
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, note.Rest):
                notes.append('rest')
            # Se l'elemento è un accordo (più note insieme)
            elif isinstance(element, chord.Chord):
                # Codifichiamo l'accordo come stringa di ID (es: 4.7.10)
                notes.append('.'.join(str(n) for n in element.normalOrder))

    return notes


all_notes = get_notes_from_midi("data/")

Parsing: data\2_of_a_kind_jp.mid
Parsing: data\500_miles_high-Chick-Corea_ee.mid
Parsing: data\99_miles_from_l_a-kar_rt.mid
Parsing: data\aint_that_a_kick_in_the_head_r2_rt.mid
Parsing: data\aint_we_got_fun_bz2-bz3.mid
Parsing: data\aja_sr3.mid
Parsing: data\alley_cat_bb10.mid
Parsing: data\all_blues-miles-davis_bl.mid
Parsing: data\all_blues-Miles-Davis_dz.mid
Parsing: data\all_my_tomorrows-kar-kos_mw.mid
Parsing: data\all_night_long_melod.mid


C:\Users\andre\PycharmProjects\corso_ai\.venv\Lib\site-packages\music21\midi\translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=13, data=b'\xa9 1931 Francis, Day & Hunter Ltd. London'>; getting generic Instrument
  warnings.warn(


Parsing: data\all_of_me-1931-vs2-kar_jpp.mid
Parsing: data\all_of_you_mw.mid
Parsing: data\all_the_things_you_are-1940-Kern-Mayerl_jpp.mid
Parsing: data\all_the_things_you_are-2_dm.mid


C:\Users\andre\PycharmProjects\corso_ai\.venv\Lib\site-packages\music21\midi\translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=14, data=b'\xa9 1957 by The International Music Network Ltd. London'>; getting generic Instrument
  warnings.warn(


Parsing: data\all_the_way-1957-vs2-kar_jpp.mid
Parsing: data\all_the_way_mw.mid
Parsing: data\almost_like_being_in_love_gw.mid
Parsing: data\alone_together1-MDavis_bl.mid
Parsing: data\alone_together2-MDavis_bl.mid
Parsing: data\always_and_forever_bnzo.mid
Parsing: data\amor_de_roca_cd.mid
Parsing: data\a_cottage_for_sale_rs.mid
Parsing: data\a_day_in_the_life_of_a_fool_jhall.mid
Parsing: data\a_fine_romance_mw.mid
Parsing: data\a_foggy_day_r_gw.mid
Parsing: data\original_metheny.mid


In [122]:
# Otteniamo tutti i nomi delle note uniche
pitchnames = sorted(set(item for item in all_notes))
vocab_size = len(pitchnames)

# Creiamo un dizionario per mappare le note a numeri
note_to_int = {note: number for number, note in enumerate(pitchnames)}

# Prepariamo gli input (sequenze) e gli output (la nota successiva)
sequence_length = 100
network_input = []
network_output = []

for i in range(0, len(all_notes) - sequence_length):
    sequence_in = all_notes[i:i + sequence_length]
    sequence_out = all_notes[i + sequence_length]

    network_input.append([note_to_int[char] for char in sequence_in])
    network_output.append(note_to_int[sequence_out])

print(note_to_int)

{'0': 0, '0.1': 1, '0.1.2.6': 2, '0.1.3.5.8': 3, '0.1.4.7': 4, '0.1.5.7': 5, '0.1.5.8': 6, '0.2': 7, '0.2.3.5': 8, '0.2.3.7': 9, '0.2.4.6.9': 10, '0.2.4.7': 11, '0.2.4.7.9': 12, '0.2.5': 13, '0.2.6': 14, '0.2.6.8': 15, '0.2.7': 16, '0.3': 17, '0.3.5': 18, '0.3.5.7.8': 19, '0.3.5.8': 20, '0.3.6.9': 21, '0.3.7': 22, '0.4': 23, '0.4.6': 24, '0.4.7': 25, '0.4.8': 26, '0.5': 27, '0.5.6': 28, '0.6': 29, '1': 30, '1.2.3.6.7': 31, '1.3': 32, '1.3.5.8': 33, '1.3.7': 34, '1.4': 35, '1.4.6.8.9': 36, '1.4.6.9': 37, '1.4.7': 38, '1.4.7.10': 39, '1.4.8': 40, '1.5': 41, '1.5.7': 42, '1.5.8': 43, '1.6': 44, '1.6.7': 45, '1.7': 46, '10': 47, '10.0': 48, '10.0.2.4': 49, '10.0.2.4.7': 50, '10.0.2.5': 51, '10.0.2.5.7': 52, '10.0.3.6': 53, '10.0.5': 54, '10.1': 55, '10.1.3.6': 56, '10.1.4': 57, '10.1.5': 58, '10.11': 59, '10.11.0': 60, '10.11.3.6': 61, '10.2': 62, '10.2.4': 63, '10.2.4.5': 64, '10.2.5': 65, '10.3': 66, '11': 67, '11.0': 68, '11.0.2.4.7': 69, '11.0.4.5': 70, '11.0.4.7': 71, '11.1': 72, '11.

In [123]:
import torch.nn as nn

class JazzLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers):
        super(JazzLSTM, self).__init__()
        # L'Embedding trasforma un numero (es. nota 42) in un vettore denso
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # La LSTM vera e propria
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=0.3)

        # Lo strato finale che decide qual è la prossima nota
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x shape: [batch_size, seq_len]
        x = self.embedding(x) # shape: [batch_size, seq_len, embed_dim]

        # out: tutti gli stati nascosti, _ : l'ultimo stato (h, c)
        out, _ = self.lstm(x)

        # Prendiamo solo l'output dell'ultimo step temporale
        out = self.fc(out[:, -1, :])
        return out

In [124]:
import torch
from torch.utils.data import Dataset, DataLoader

class MusicDataset(Dataset):
    def __init__(self, inputs, targets):
        # Convertiamo le liste in tensori di tipo Long (interi)
        self.inputs = torch.tensor(inputs, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# Parametri di addestramento
BATCH_SIZE = 64  # Quante sequenze vede la rete prima di aggiornare i pesi

# Inizializziamo Dataset e DataLoader
dataset = MusicDataset(network_input, network_output)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Test rapido: prendiamo un batch
data_iter = iter(train_loader)
sample_input, sample_target = next(data_iter)

print(f"Forma del batch input: {sample_input.shape}")   # [64, 100]
print(f"Forma del batch target: {sample_target.shape}") # [64]

Forma del batch input: torch.Size([64, 100])
Forma del batch target: torch.Size([64])


In [125]:
# Controlliamo se la GPU è disponibile
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sto addestrando su: {device}")

# Iperparametri
VOCAB_SIZE = len(pitchnames)
EMBED_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2

# Inizializziamo il modello
model = JazzLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS).to(device)



Sto addestrando su: cuda


In [126]:
# Perdita e Ottimizzatore
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [131]:
EPOCHS = 50 # Quante volte la rete vedrà l'intero dataset
LR = 0.001 # Learning Rate

model.train() # Mettiamo il modello in modalità addestramento
for epoch in range(EPOCHS):
    total_loss = 0

    for batch_idx, (sequences, targets) in enumerate(train_loader):
        # Spostiamo i dati sulla GPU/CPU
        sequences, targets = sequences.to(device), targets.to(device)

        # 1. Reset dei gradienti (fondamentale in PyTorch!)
        optimizer.zero_grad()

        # 2. Forward pass: la rete fa la sua previsione
        outputs = model(sequences)

        # 3. Calcolo dell'errore
        loss = criterion(outputs, targets)

        # 4. Backward pass: calcolo dei gradienti
        loss.backward()

        # 5. Ottimizzazione: aggiornamento dei pesi
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

Epoch [1/50], Loss: 0.4626
Epoch [2/50], Loss: 0.4311
Epoch [3/50], Loss: 0.4202
Epoch [4/50], Loss: 0.4026
Epoch [5/50], Loss: 0.3908
Epoch [6/50], Loss: 0.4033
Epoch [7/50], Loss: 0.3668
Epoch [8/50], Loss: 0.3497
Epoch [9/50], Loss: 0.3387
Epoch [10/50], Loss: 0.3264
Epoch [11/50], Loss: 0.3116
Epoch [12/50], Loss: 0.2998
Epoch [13/50], Loss: 0.2911
Epoch [14/50], Loss: 0.2904
Epoch [15/50], Loss: 0.2758
Epoch [16/50], Loss: 0.2731
Epoch [17/50], Loss: 0.2555
Epoch [18/50], Loss: 0.2416
Epoch [19/50], Loss: 0.2367
Epoch [20/50], Loss: 0.2250
Epoch [21/50], Loss: 0.2444
Epoch [22/50], Loss: 0.2148
Epoch [23/50], Loss: 0.2016
Epoch [24/50], Loss: 0.1942
Epoch [25/50], Loss: 0.1908
Epoch [26/50], Loss: 0.1764
Epoch [27/50], Loss: 0.1730
Epoch [28/50], Loss: 0.1682
Epoch [29/50], Loss: 0.1607
Epoch [30/50], Loss: 0.1647
Epoch [31/50], Loss: 0.1587
Epoch [32/50], Loss: 0.1678
Epoch [33/50], Loss: 0.1527
Epoch [34/50], Loss: 0.1319
Epoch [35/50], Loss: 0.1250
Epoch [36/50], Loss: 0.1262
E

In [116]:
import numpy as np
import torch.nn.functional as F

def generate_notes(model, network_input, pitchnames, n_generate=500, temperature=1.0):
    model.eval() # Modalità valutazione

    # Mapping inverso: da numero a nota
    int_to_note = {number: note for number, note in enumerate(pitchnames)}

    # Scegliamo una sequenza casuale come seme iniziale
    start = np.random.randint(0, len(network_input)-1)
    pattern = network_input[start]
    prediction_output = []

    # Generiamo n note
    with torch.no_grad():
        for note_index in range(n_generate):
            prediction_input = torch.tensor([pattern], dtype=torch.long).to(device)

            # Forward pass
            output = model(prediction_input)

            # Applichiamo la Temperatura
            # Più T è alta, più la distribuzione diventa "piatta" (caotica)
            output = output / temperature
            prob_dist = F.softmax(output, dim=1).cpu().numpy().reshape(-1)

            # Scegliamo la prossima nota in base alla distribuzione di probabilità
            predicted_index = np.random.choice(len(pitchnames), p=prob_dist)

            # Convertiamo l'indice in nota e salviamo
            result = int_to_note[predicted_index]
            prediction_output.append(result)

            # Aggiorniamo il pattern: togliamo la prima nota e aggiungiamo la nuova
            pattern.append(predicted_index)
            pattern = pattern[1:]

    return prediction_output

In [118]:
from music21 import stream, note, chord, instrument

def create_midi(prediction_output, filename='jazz_output.mid'):
    offset = 0
    output_notes = []

    for pattern in prediction_output:
        # 1. GESTIONE DELLE PAUSE
        if pattern == 'R' or pattern == 'rest':
            new_element = note.Rest()
            new_element.offset = offset
            output_notes.append(new_element)

        # 2. GESTIONE DEGLI ACCORDI (es: '0.4.7' o '10')
        elif ('.' in pattern) or pattern.isdigit():
            notes_in_chord = pattern.split('.')
            notes = []
            for current_note in notes_in_chord:
                new_note = note.Note(int(current_note))
                #new_note.storedInstrument = instrument.BaritoneSaxophone()
                notes.append(new_note)
            new_element = chord.Chord(notes)
            new_element.offset = offset
            output_notes.append(new_element)

        # 3. GESTIONE DELLE NOTE SINGOLE (es: 'C4')
        else:
            new_element = note.Note(pattern)
            new_element.offset = offset
            #new_element.storedInstrument = instrument.BaritoneSaxophone()
            output_notes.append(new_element)

        # Aumentiamo l'offset (la posizione nel tempo)
        # 0.5 equivale a una croma (ottavo) in un tempo standard
        offset += 0.5

    midi_stream = stream.Stream(output_notes)
    midi_stream.write('midi', fp=filename)
    print(f"File salvato con successo: {filename}")

In [134]:
# 1. Genera la sequenza (usa T=0.8 per un jazz "stabile" o T=1.2 per "improvvisazione spinta")
generated_notes = generate_notes(model, network_input, pitchnames, temperature=0.7)
print(generated_notes[:100])
# 2. Crea il file MIDI


['D4', 'rest', 'rest', '11.0.4.7', 'rest', '11.0.4.7', 'rest', '4.6.9.0', 'rest', '2.6.9', 'C4', 'rest', 'rest', '2.6.9', 'rest', '11.2.4.7', 'rest', 'rest', '4.7.9.11.0', 'rest', '4.7.9.11.0', 'rest', '0.2.4.7', 'rest', 'rest', '0.2.4.7', 'rest', '4.5.7.11', 'A2', 'rest', '1.4.6.8.9', 'rest', '1.4.6.8.9', 'rest', '9.11.0.3.6', 'rest', '11.2.4.6.7', 'rest', '11.2.4.6.7', 'rest', '7.9.10.1.3', 'rest', '9.0.2.4.5', 'rest', '4.5.7.9.11', 'rest', '11.0.2.4.7', 'rest', 'rest', 'rest', '7.9.11.2', 'rest', '7.9.2', 'rest', '7.8.10', 'D4', 'E-4', 'rest', 'D4', 'rest', 'rest', '11.0.4.7', 'rest', '11.0.4.7', 'rest', '4.6.9.0', 'rest', '2.6.9', 'C4', 'rest', 'rest', '2.6.9', 'rest', '11.2.4.7', 'rest', 'rest', '4.7.9.11.0', 'rest', '4.7.9.11.0', 'rest', '0.2.4.7', 'rest', 'rest', '0.2.4.7', 'rest', '4.5.7.11', 'A2', 'rest', '1.4.6.8.9', 'rest', '1.4.6.8.9', 'rest', '9.11.0.3.6', 'rest', '11.2.4.6.7', 'rest', '11.2.4.6.7', 'rest', '7.9.11.1.3', 'rest']


In [135]:
create_midi(generated_notes)

File salvato con successo: jazz_output.mid
